<a href="https://colab.research.google.com/github/Fahadqureshi0/Job-Recommendation-System/blob/main/Job_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Importing Dependencies**

In [1]:
# Data Manipulation
import numpy as np
import pandas as pd

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Data Preprocessing
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

**Data Collection & Preprocessing**

In [2]:
jobs_dataset = pd.read_csv('/content/all_job_post.csv')

In [3]:
# Printing 5 rows & columns
jobs_dataset.head()

,job_id,category,job_title,job_description,job_skill_set
0,3902668440,HR,Sr Human Resource Generalist,SUMMARY\nTHE SR. HR GENERALIST PROVIDES HR EXP...,"['employee relations', 'talent acquisition', '..."
1,3905823748,HR,Human Resources Manager,BE PART OF A STELLAR TEAM AT YSB AS THE MANAGE...,"['Talent Acquisition', 'Employee Performance M..."
2,3905854799,HR,Director of Human Resources,OUR CLIENT IS A THRIVING ORGANIZATION OFFERING...,"['Human Resources Management', 'Recruitment', ..."
3,3905834061,HR,Chief Human Resources Officer,JOB TITLE: CHIEF HUMAN RESOURCES OFFICER (CHRO...,"['talent management', 'organizational developm..."
4,3906250451,HR,Human Resources Generalist (Hybrid Role),DESCRIPTION\n\n WHO WE ARE \n\nAVI-SPL IS A DI...,"['Microsoft Office', 'Data analysis', 'Employe..."


In [4]:
# Last columns
jobs_dataset.tail()

,job_id,category,job_title,job_description,job_skill_set
1162,3905299905,BUSINESS-DEVELOPMENT,Intern - Business Development,REQUIREMENTS\n\n DESCRIPTION & REQUIREMENTS \n...,"['MS Office Suite', 'PowerPoint', 'Excel', 're..."
1163,3885829894,BUSINESS-DEVELOPMENT,Business Development Representative,IT'S FUN TO WORK IN A COMPANY WHERE PEOPLE TRU...,"['Collaboration', 'Communication', 'Problem So..."
1164,3901649881,BUSINESS-DEVELOPMENT,Enterprise Business Development Representative...,JOIN OUR DYNAMIC AI TEAM AS AN ENTERPRISE BUSI...,"['Salesforce', 'data analysis', 'lead generati..."
1165,3904049863,BUSINESS-DEVELOPMENT,Senior Director Business Development,ROOM 8 GROUP IS THE WORLD’S FASTEST GROWING ST...,"['business development', 'sales', 'corporate s..."
1166,3904084138,BUSINESS-DEVELOPMENT,Global Business Development Partner - 100% Remote,EXCITING OPPORTUNITY: GLOBAL BUSINESS DEVELOPM...,"['strategic marketing', 'business development'..."


In [5]:
# Shape of Datset
jobs_dataset.shape

(1167, 5)

In [6]:
# Information about Dataset
jobs_dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1167 entries, 0 to 1166
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   job_id           1167 non-null   int64 
 1   category         1167 non-null   object
 2   job_title        1167 non-null   object
 3   job_description  1167 non-null   object
 4   job_skill_set    1167 non-null   object
dtypes: int64(1), object(4)
memory usage: 45.7+ KB


In [7]:
# Check for null values
jobs_dataset.isnull().sum()

,0
job_id,0
category,0
job_title,0
job_description,0
job_skill_set,0


In [9]:
# Combiining Features
jobs_dataset['job_text'] = (jobs_dataset['job_title'] + " " +jobs_dataset['job_description'] + " " +['job_skill_set'])

In [10]:
jobs_dataset[['job_title', 'job_text']]

,job_title,job_text
0,Sr Human Resource Generalist,Sr Human Resource Generalist SUMMARY\nTHE SR. ...
1,Human Resources Manager,Human Resources Manager BE PART OF A STELLAR T...
2,Director of Human Resources,Director of Human Resources OUR CLIENT IS A TH...
3,Chief Human Resources Officer,Chief Human Resources Officer JOB TITLE: CHIEF...
4,Human Resources Generalist (Hybrid Role),Human Resources Generalist (Hybrid Role) DESCR...
...,...,...
1162,Intern - Business Development,Intern - Business Development REQUIREMENTS\n\n...
1163,Business Development Representative,Business Development Representative IT'S FUN T...
1164,Enterprise Business Development Representative...,Enterprise Business Development Representative...
1165,Senior Director Business Development,Senior Director Business Development ROOM 8 GR...


**TF-IDF Vectorization**

In [24]:
tfidf = TfidfVectorizer(stop_words='english')

In [43]:
# Converting Text-Data to vectors
job_text_vectors = tfidf.fit_transform(jobs_dataset['job_text'])

In [26]:
print(job_text_vectors.shape)

(1167, 20988)


**Test Resume**

In [27]:
resume = """
Machine Learning Engineer with experience in Python,
machine learning, data analysis and SQL.

Skills:
Python
Machine Learning
Scikit-learn
Pandas
NumPy
SQL
TensorFlow
Data Analysis

Experience:
Developed machine learning models for classification
and regression problems.
"""

In [28]:
# Transforming Resume Data into vectors
resume_vector = tfidf.transform([resume])

**Cosine Similarity**

In [29]:
similarity_score =  cosine_similarity(resume_vector, job_text_vectors)

In [30]:
print(similarity_score.shape)

(1, 1167)


**Getting Top Best Jobs**

In [31]:
# Jobs Score
job_score = similarity_score[0]

In [32]:
# Finding top 10 jobs:
top_jobs = job_score.argsort()[::-1][:10]

In [33]:
# Recommendations
recommendations = jobs_dataset.iloc[top_jobs].copy()

In [34]:
recommendations['similarity_score'] = job_score[top_jobs]

In [35]:
recommendations[[
    "job_title",
    "category",
    "similarity_score"
]]

,job_title,category,similarity_score
892,Finance Learning Advisor,FINANCE,0.195354
293,Information Technology Specialist,INFORMATION-TECHNOLOGY,0.149496
832,"VP / Client, Finance & Strategy Data Scientist...",FINANCE,0.134614
757,Staff Data Engineer - People & Finance Tech (R...,FINANCE,0.128151
897,Lead Software Engineer - Finance (Hybrid),FINANCE,0.122923
269,Information Technology Engineer,INFORMATION-TECHNOLOGY,0.116496
1164,Enterprise Business Development Representative...,BUSINESS-DEVELOPMENT,0.115192
831,Data Solutions Analyst II (Managed Care - Fina...,FINANCE,0.110467
770,Finance Product Data Owner,FINANCE,0.102658
435,Information Technology Field Engineer,INFORMATION-TECHNOLOGY,0.101082


**Recommendation System**

In [36]:
def recommend_jobs(resume):

    # Convert resume into TF-IDF vector
    resume_vector = tfidf.transform([resume])

    # Calculate similarity between resume and all jobs
    similarity_scores = cosine_similarity(
        resume_vector,
        job_text_vectors
    )

    # Get scores for the resume
    job_scores = similarity_scores[0]

    # Sort jobs from highest similarity to lowest
    sorted_jobs = sorted(enumerate(job_scores),key=lambda x: x[1],reverse=True)

    print("Jobs Suggested For You:\n")

    i = 1

    for job in sorted_jobs:

        index = job[0]
        score = job[1]

        job_title = jobs_dataset.iloc[index]["job_title"]
        category = jobs_dataset.iloc[index]["category"]

        print(i,".",job_title,"| Category:",category,"| Match:",round(score * 100, 2),"%")

        i += 1

        if i > 10:
            break

In [38]:
# User Input
resume = input("Enter your resume: ")

# Recommendation
recommend_jobs(resume)

Enter your resume: Finance Analyst with 3 years of experience in financial analysis, budget planning, investment research, and business reporting.  Skills: Financial Analysis Microsoft Excel Financial Modeling Budget Forecasting SQL Power BI Accounting Risk Analysis Valuation Corporate Finance  Experience: Prepared monthly financial reports and budget forecasts for senior management. Built financial models to evaluate business performance and investment opportunities. Analyzed revenue, expenses, and profitability using Excel and SQL. Created interactive dashboards in Power BI to support financial decision-making. Collaborated with accounting and operations teams to improve cost efficiency.  Education: Bachelor of Business Administration (Finance)  Certifications: Financial Modeling & Valuation Microsoft Excel Advanced
Jobs Suggested For You:

1 . Finance Manager | Category: FINANCE | Match: 40.01 %
2 . Finance Analyst (Job #247) ($77,056 - $85,028) | Category: FINANCE | Match: 39.18 %


**Saving Recommendation Systems**

In [39]:
# Importing Pickle
import pickle

In [40]:
# Dataset File
with open('jobs_dataset.pkl', 'wb') as file:
    pickle.dump(jobs_dataset, file)

In [41]:
# Jobs Text Vectorizers
with open('job_text_vectors.pkl', 'wb') as file:
    pickle.dump(job_text_vectors, file)

In [42]:
# TF-IDF Vectorizers
with open('tfidf.pkl', 'wb') as file:
    pickle.dump(tfidf, file)